# Rationale: 
> Mapping between source and target vocabulary concepts. Additional function of suggestion of concept placement in target vocabulary when mapping threshold is not met. 

USE CASE: VO : RxNorm mapping
1. Phase 0: Environment Setup
    1. Elastisearch Setup
    1. FAISS Setup
1. Phase 1: SIMILARITY
    1. Lexical Similarity (BM25 Indexing)
    1. Semantic Similarity (Dense Indexing - Faiss Inner Product ~ cosine)
    1. Reciprocal Ranked Fusion of similarity scores (RRF) to generate candidates
1. Phase 2: LLM REFINEMENT
    1. The output would be either a match from the pool or 'NO MATCH'
    1. If output is 'NO MATCH' an LLM agent would take over
1. Phase 3: LLM AGENT
    1. Agent Tools
        1. Vocabulary traversal
        1. Similarity fromo Phase 1
    1. Agent would then suggest/place a new concept at location

In [5]:
MODEL_ID_LIST = ['tavakolih/all-MiniLM-L6-v2-pubmed-full', 'Alibaba-NLP/gte-Qwen2-1.5B-instruct', 'google/medgemma-27b-text-it']

# Phase 0: ENvironment Setup

## 1.1 Elastisearch Setup

In [2]:
from owlready2 import *
from typing import List, Sequence, Tuple, Optional, Callable, Union
import numpy as np
import faiss
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import torch
from sentence_transformers import SentenceTransformer
import requests
import pandas as pd
import pickle
import json
import pandas as pd
from tqdm import tqdm
import subprocess
import os
import signal
import time
from pathlib import Path
import time, requests, subprocess
from pathlib import Path

In [7]:
def load_model(model_id):
    print(f"Loading model {model_id}...")
    model = SentenceTransformer(model_id, device='cuda')
    print(f"Model {model_id} loaded to CUDA {model.device} successfully.")
    return model

### Dense Index Functions:

In [8]:
# encode texts with the model with parameter for batch size
def encode_texts(model, texts, batch_size=32, normalize=True):
    embeddings = model.encode(
        texts, 
        batch_size=batch_size,
        show_progress_bar=True, 
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    )
    return embeddings.astype("float32") # FAISS needs float32

In [9]:
#build faiss index
def build_faiss_index(embeddings:np.ndarray,ids: Optional[Sequence[int]] = None,use_gpu: bool = True) -> faiss.Index:
    import faiss
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product of normalized vectors is  ~ cosine similarity
    if use_gpu and faiss.get_num_gpus() > 0:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    if ids is not None:
        id_map = faiss.IndexIDMap(index)
        id_map.add_with_ids(embeddings, np.asarray(ids, dtype="int64"))
        print(f"Index built with {len(ids)} IDs.")
        return id_map
    else:
        index.add(embeddings)
        print(f"Index built with {embeddings.shape[0]} vectors.")
        return index

In [10]:
def build_and_save_dense_index(
    df: pd.DataFrame, 
    model, 
    label_column: str = "label",
    batch_size: int = 64,
    normalize: bool = True,
    use_gpu: bool = True,
    save_index: bool = True,
    index_filename: str = "vo_faiss.bin"
) -> faiss.Index:
    
    print(f"Encoding {len(df)} texts from column '{label_column}'...")
    
    # Extract texts and encode
    texts = df[label_column].tolist()
    vecs = encode_texts(model, texts, batch_size=batch_size, normalize=normalize)
    
    # Use DataFrame index as FAISS IDs
    faiss_ids = df.index.to_numpy(dtype="int64")
    
    # Build FAISS index
    faiss_index = build_faiss_index(vecs, ids=faiss_ids, use_gpu=use_gpu)
    
    # Optionally save to disk
    if save_index:
        print(f"Saving index to {index_filename}...")
        faiss_cpu = faiss.index_gpu_to_cpu(faiss_index) if use_gpu else faiss_index
        faiss.write_index(faiss_cpu, index_filename)
        print(f"Index saved successfully to {index_filename}")
    
    return faiss_index

### BM25 Index

#### 1. ElastiSearch Env Load

In [11]:
ES_HOME = Path("/data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0")
ES_PORT  = 9200
# PID_FILE  = ES_HOME / "run/es.pid"
# LOG_FILE  = ES_HOME / "logs/console.log"

In [14]:
def run_elasticsearch():
    # Path to elasticsearch executable
    es_executable = ES_HOME / "bin/elasticsearch"
    
    # Environment variables for Elasticsearch
    env = os.environ.copy()
    env.update({
        "ES_JAVA_OPTS": "-Xms2g -Xmx2g",
        "discovery.type": "single-node",
        "xpack.security.enabled": "false"
    })
    
    print(f"Starting Elasticsearch from {es_executable}...")
    
    # Run Elasticsearch in background
    process = subprocess.Popen(
        [str(es_executable)],
        env=env,
        cwd=ES_HOME,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    # Wait for Elasticsearch to start
    print(f"Elasticsearch started with PID: {process.pid}")
    print("Waiting for service to become available...")
    
    # Try connecting to verify it's up
    for _ in range(60):
        try:
            response = requests.get(f"http://localhost:{ES_PORT}", timeout=2)
            if response.status_code == 200:
                print(f"Elasticsearch is running: {response.json().get('version', {}).get('number')}")
                return process
        except (requests.ConnectionError, requests.Timeout):
            print(".", end="", flush=True)
            time.sleep(2)
    
    print("Elasticsearch didn't start properly within the timeout period")
    return process

# Run it
es_process = run_elasticsearch()

Starting Elasticsearch from /data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0/bin/elasticsearch...
Elasticsearch started with PID: 433112
Waiting for service to become available...
Elasticsearch is running: 9.0.0


In [13]:
es_process.terminate()
time.sleep(5)
if es_process.poll() is None:  # Still running
    print("Sending SIGKILL...")
    es_process.kill()

Sending SIGKILL...


In [15]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200")
es.info()

ObjectApiResponse({'name': 'msdplaplp001.uth.tmc.edu', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'm4zPSJyjTKCUH2PiE0ADfw', 'version': {'number': '9.0.0', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '112859b85d50de2a7e63f73c8fc70b99eea24291', 'build_date': '2025-04-08T15:13:46.049795831Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

Refs: 
* Analyzer [https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/analyzer]
* Syns graph [https://www.elastic.co/docs/reference/text-analysis/analysis-synonym-graph-tokenfilter]
* Similarity [https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity]
* Mapping []

Todo: Apply synonyms during indexing or search ?


In [16]:
VO_INDEX = "vo_bm25"

try:
    response = es.indices.delete(index=VO_INDEX)
    print(f"Index '{VO_INDEX}' deleted successfully: {response}")
except Exception as e:
    print(f"Error deleting index '{VO_INDEX}': {e}")

vo_settings = {
  "settings": {
    "analysis": {
      "filter": {
        "vaccine_syns": {                
          "type": "synonym_graph",
          "synonyms_path": "vaccine_syns.txt"
        }
      },
      "analyzer": {
        "vaccine_analyzer": {
          "tokenizer": "standard",
          "filter": ["lowercase", "asciifolding", "vaccine_syns"]
        }
      }
    },
    "similarity": {
      "bm25_short": {
        "type": "BM25",
        "k1": 0.9,
        "b":  0.4
      }
    }
  },
  "mappings": {
    "properties": {
      "vo_id":  { "type": "keyword" },        
      "iri":    { "type": "keyword" },        
      "text": {
        "type": "text",
        "analyzer":   "vaccine_analyzer",
        "similarity": "bm25_short"
      }
    }
  }
}

es.indices.create(index=VO_INDEX, body=vo_settings, ignore=400)


Index 'vo_bm25' deleted successfully: {'acknowledged': True}


/tmp/ipykernel_138352/459341731.py:46: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=VO_INDEX, body=vo_settings, ignore=400)


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'vo_bm25'})

#### 2. Loading RxNorm via RxNav API

#### 2.1 Helper Functions  

In [17]:
from tqdm import tqdm
import concurrent.futures
import time
from threading import Lock
from functools import wraps

In [18]:
class RateLimiter:
    def __init__(self, max_calls_per_second=5):
        self.max_calls_per_second = max_calls_per_second
        self.min_interval = 1.0 / max_calls_per_second
        self.last_called = 0
        self.lock = Lock()
    
    def wait(self):
        with self.lock:
            elapsed = time.time() - self.last_called
            left_to_wait = self.min_interval - elapsed
            if left_to_wait > 0:
                time.sleep(left_to_wait)
            self.last_called = time.time()

rate_limiter = RateLimiter(max_calls_per_second=10)

In [19]:
def rate_limited(rate_limiter: RateLimiter):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            rate_limiter.wait()
            return func(*args, **kwargs)
        return wrapper
    return decorator

In [27]:
def threaded_params_fetch(
    params_list: List[dict],
    # apikey: str,
    fetch_func: Callable[[dict, str], Union[dict, List[dict]]],
    desc: str = "Fetching data",
    max_workers: int = 10
):
    """
    Specialized version of threaded_umls_fetch for parameter dictionaries
    """
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(fetch_func, params): str(params)
            for params in params_list
        }
        with tqdm(total=len(params_list), desc=desc) as pbar:
            for future in concurrent.futures.as_completed(futures):
                params_str = futures[future]
                try:
                    result = future.result()
                    if isinstance(result, list):
                        results.extend(result)
                    else:
                        results.append(result)
                    pbar.set_postfix({"Current": params_str[:50]})
                except Exception as e:
                    print(f"Error fetching {params_str}: {e}")
                pbar.update(1)
    return results

In [3]:
# Helper functions from rxnorm_getter 
# Todo: Move them out of the identify_missing_rxnorm_terms function: 

def get_rxnorm_data(rxcui):
        result = {}
        result['rxcui'] = rxcui
        status_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/historystatus.json?"
        try:
            response = requests.get(status_url)
            if response.status_code == 200:
                data = response.json()
                result['name'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("name")
                result['status'] = data.get("rxcuiStatusHistory", {}).get("metaData", {}).get("status")
                result['tty'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("tty")
        except requests.RequestException as e:
            print(f"Error fetching RxNorm status for {rxcui}: {e}")
        return result

def get_class_descendants(class_id, class_type):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classTree.json?classId={class_id}&classType={class_type}"
    descendants = []
    
    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        descendants = _collect_leaf_classids(data["rxclassTree"])
        
    except Exception as e:
        print(f"API error with {class_id}: {e}")
        
    return descendants

def _collect_leaf_classids(tree):
    leaves = []
    for node in tree:
        cid = node["rxclassMinConceptItem"]["classId"]
        cname = node["rxclassMinConceptItem"]["className"]
        children = node.get("rxclassTree")
        if children:                      
            leaves.extend(_collect_leaf_classids(children))
        else:                             
            leaves.append({'ID': cid, 'Name': cname})
    return leaves

def get_rx_class_members(params):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classMembers.json"
    class_id = params.get("classId")
    source = params.get("relaSource")
    result = {"class_id": class_id, "source": source, "related_ids": None}
    
    try:
        response = requests.get(api_url, params=params)
        response.raise_for_status()      
        data = response.json()
        if "drugMemberGroup" in data and "drugMember" in data["drugMemberGroup"]:
            related_ids = [item["minConcept"]["rxcui"] for item in data["drugMemberGroup"]["drugMember"]]
            result["related_ids"] = related_ids
    except Exception as e:
        print(f"API error with {class_id}: {e}")

    return result

def get_related_rxcui(rxcui):
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.json"
    result = {"rxcui": rxcui, "related_ids": None}

    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        concept_groups = data.get("allRelatedGroup", {}).get("conceptGroup", [])
        related_concepts = []
        for group in concept_groups:
            if "conceptProperties" in group:
                related_concepts.extend([concept.get("rxcui") for concept in group["conceptProperties"]])

        result["related_ids"] = related_concepts

    except Exception as e:
        print(f"API error with {rxcui}: {e}")

    return result

def get_all_obsolete_concepts():
    api_url = "https://rxnav.nlm.nih.gov/REST/allstatus.json?status=obsolete"
    obsolete_concepts = []

    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        concepts = data.get('minConceptGroup', {}).get('minConcept', [])
        for concept in concepts:
            rxcui = concept.get('rxcui')
            name = concept.get('name')
            tty = concept.get('tty')
            obsolete_concepts.append({
                'rxcui': rxcui,
                'name': name,
                'tty': tty
            })
        print(f"Found {len(obsolete_concepts)} obsolete concepts.")

    except Exception as e:
        print(f"API error: {e}")

    return obsolete_concepts

In [21]:
@rate_limited(rate_limiter)
def get_rxnorm_data(rxcui: str, apikey: str) -> dict:
    """Get RxNorm data for a given RXCUI"""
    result = {}
    result['rxcui'] = rxcui
    status_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/historystatus.json"
    
    try:
        response = requests.get(status_url, params={'apikey': apikey} if apikey else {})
        if response.status_code == 200:
            data = response.json()
            result['name'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("name")
            result['status'] = data.get("rxcuiStatusHistory", {}).get("metaData", {}).get("status")
            result['tty'] = data.get("rxcuiStatusHistory", {}).get("attributes", {}).get("tty")
    except requests.RequestException as e:
        print(f"Error fetching RxNorm status for {rxcui}: {e}")
    
    return result

@rate_limited(rate_limiter)
def get_class_descendants_single(class_id: str, apikey: str, class_type: str = "ATC1-4") -> List[dict]:
    """Get descendants for a single class ID"""
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classTree.json?classId={class_id}&classType={class_type}"
    descendants = []
    
    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        descendants = _collect_leaf_classids(data["rxclassTree"])
        
    except Exception as e:
        print(f"API error with {class_id}: {e}")
        
    return descendants

@rate_limited(rate_limiter)
def get_rx_class_members_single(class_params: dict, apikey: str) -> dict:
    """Get class members for a single class"""
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxclass/classMembers.json"
    class_id = class_params.get("classId")
    source = class_params.get("relaSource")
    result = {"class_id": class_id, "source": source, "related_ids": None}
    
    try:
        response = requests.get(api_url, params=class_params)
        response.raise_for_status()      
        data = response.json()
        if "drugMemberGroup" in data and "drugMember" in data["drugMemberGroup"]:
            related_ids = [item["minConcept"]["rxcui"] for item in data["drugMemberGroup"]["drugMember"]]
            result["related_ids"] = related_ids
    except Exception as e:
        print(f"API error with {class_id}: {e}")

    return result

@rate_limited(rate_limiter)
def get_related_rxcui_single(rxcui: str, apikey: str) -> dict:
    """Get related RXCUIs for a single RXCUI"""
    api_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.json"
    result = {"rxcui": rxcui, "related_ids": None}

    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        concept_groups = data.get("allRelatedGroup", {}).get("conceptGroup", [])
        related_concepts = []
        for group in concept_groups:
            if "conceptProperties" in group:
                related_concepts.extend([concept.get("rxcui") for concept in group["conceptProperties"]])

        result["related_ids"] = related_concepts

    except Exception as e:
        print(f"API error with {rxcui}: {e}")

    return result

@rate_limited(rate_limiter)
def get_all_obsolete_concepts_single(dummy_param: str, apikey: str) -> List[dict]:
    """Get all obsolete concepts (doesn't need threading but keeping consistent interface)"""
    api_url = "https://rxnav.nlm.nih.gov/REST/allstatus.json?status=obsolete"
    obsolete_concepts = []

    try:
        response = requests.get(api_url)
        response.raise_for_status()      
        data = response.json()
        concepts = data.get('minConceptGroup', {}).get('minConcept', [])
        for concept in concepts:
            rxcui = concept.get('rxcui')
            name = concept.get('name')
            tty = concept.get('tty')
            obsolete_concepts.append({
                'rxcui': rxcui,
                'name': name,
                'tty': tty
            })
        print(f"Found {len(obsolete_concepts)} obsolete concepts.")

    except Exception as e:
        print(f"API error: {e}")

    return obsolete_concepts

#### 2.2 Implementation

In [ ]:
# Collect RxNorm concepts from different sources
rx_concepts = set()

# ATC concepts - First get descendants, then get members
print("Collecting from ATC...")
atc_class_ids = ["J07"]  # Can be extended to multiple classes
atc_descendants_results = threaded_params_fetch(
    cui_list=atc_class_ids,
    # apikey=apikey,
    fetch_func=lambda class_id, api_key: get_class_descendants_single(class_id, api_key, "ATC1-4"),
    desc="Fetching ATC Descendants",
    max_workers=5
)

# Flatten descendants and get class members
atc_class_params = []
for descendants_list in atc_descendants_results:
    for atc in descendants_list:
        atc_class_params.append({"classId": atc['ID'], "relaSource": "ATCPROD"})

# Get class members for all ATC classes
if atc_class_params:
    atc_members_results = threaded_params_fetch(
        cui_list=atc_class_params,
        apikey=apikey,
        fetch_func=lambda params, api_key: get_rx_class_members_single(params, api_key),
        desc="Fetching ATC Class Members",
        max_workers=10
    )
    
    for result in atc_members_results:
        if result.get('related_ids'):
            rx_concepts.update(result['related_ids'])

# VA concepts
print("Collecting from VA...")
VA_Classes = ["IM100", "IM105", "IM109"]
rela = ["has_vaclass", "has_vaclass_extended"]
va_class_params = []
for va in VA_Classes:
    for r in rela:
        va_class_params.append({"classId": va, "relaSource": "VA", "rela": r})

va_members_results = threaded_params_fetch(
    cui_list=va_class_params,
    apikey=apikey,
    fetch_func=lambda params, api_key: get_rx_class_members_single(params, api_key),
    desc="Fetching VA Class Members",
    max_workers=10
)

for result in va_members_results:
    if result.get('related_ids'):
        rx_concepts.update(result['related_ids'])

# CVX concepts
print("Collecting from CVX...")
cvx_descendants_results = threaded_params_fetch(
    cui_list=["0"],
    apikey=apikey,
    fetch_func=lambda class_id, api_key: get_class_descendants_single(class_id, api_key, "CVX"),
    desc="Fetching CVX Descendants",
    max_workers=5
)

cvx_class_params = []
for descendants_list in cvx_descendants_results:
    for cvx in descendants_list:
        cvx_class_params.append({"classId": cvx['ID'], "relaSource": "CDC", "rela": "isa_CVX"})

if cvx_class_params:
    cvx_members_results = threaded_params_fetch(
        cui_list=cvx_class_params,
        apikey=apikey,
        fetch_func=lambda params, api_key: get_rx_class_members_single(params, api_key),
        desc="Fetching CVX Class Members",
        max_workers=10
    )
    
    for result in cvx_members_results:
        if result.get('related_ids'):
            rx_concepts.update(result['related_ids'])

# DailyMed concepts
print("Collecting from DailyMed...")
dm_classes = ["N0000193912", "D014612"]
rela = ["has_epc", "has_chemical_structure"]
dm_class_params = []
for dm in dm_classes:
    for r in rela:
        dm_class_params.append({"classId": dm, "relaSource": "DAILYMED", "rela": r})

dm_members_results = threaded_params_fetch(
    cui_list=dm_class_params,
    apikey=apikey,
    fetch_func=lambda params, api_key: get_rx_class_members_single(params, api_key),
    desc="Fetching DailyMed Class Members",
    max_workers=10
)

for result in dm_members_results:
    if result.get('related_ids'):
        rx_concepts.update(result['related_ids'])

print(f"Found {len(rx_concepts)} initial RxNorm concepts")

# Get related concepts
print("Finding related concepts...")
all_rx_list = list(rx_concepts)
related_results = threaded_params_fetch(
    cui_list=all_rx_list,
    apikey=apikey,
    fetch_func=get_related_rxcui_single,
    desc="Fetching Related RXCUIs",
    max_workers=15
)

for result in related_results:
    if result.get('related_ids'):
        all_rx_list.extend(result['related_ids'])

all_rx_final = list(set(all_rx_list))
print(f"Found {len(all_rx_final)} total RxNorm concepts after expansion")

# Get RxNorm data for all concepts
print("Pulling RxNorm data for identified terms...")
rxnav_data = threaded_params_fetch(
    cui_list=all_rx_final,
    apikey=apikey,
    fetch_func=get_rxnorm_data,
    desc="Fetching RxNorm Data",
    max_workers=15
)

rxnav_df = pd.DataFrame(rxnav_data)
rxnav_df.rename(columns={"name": "label"}, inplace=True)
rxnav_df.to_pickle("rxnav_data.pkl")
print(f"Saved {len(rxnav_df)} RxNorm records")

In [5]:
# Collect RxNorm concepts from different sources
rx_concepts = set()

# ATC concepts
print("Collecting from ATC...")
atc_descendants = get_class_descendants("J07", "ATC1-4")
for atc in atc_descendants:
    params = {"classId": atc['ID'], "relaSource": "ATCPROD"}
    res = get_rx_class_members(params)
    if res['related_ids']:
        rx_concepts.update(res['related_ids'])

# VA concepts
print("Collecting from VA...")
VA_Classes = ["IM100", "IM105", "IM109"]
rela = ["has_vaclass", "has_vaclass_extended"]
for va in VA_Classes:
    for r in rela:
        params = {"classId": va, "relaSource": "VA", "rela": r}
        res = get_rx_class_members(params)
        if res['related_ids']:
            rx_concepts.update(res['related_ids'])

# CVX concepts
print("Collecting from CVX...")
cvx_descendants = get_class_descendants("0", "CVX")
for cvx in cvx_descendants:
    params = {"classId": cvx['ID'], "relaSource": "CDC", "rela": "isa_CVX"}
    res = get_rx_class_members(params)
    if res['related_ids']:
        rx_concepts.update(res['related_ids'])

# DailyMed concepts
print("Collecting from DailyMed...")
dm_classes = ["N0000193912", "D014612"]
rela = ["has_epc", "has_chemical_structure"]
for dm in dm_classes:
    for r in rela:
        params = {"classId": dm, "relaSource": "DAILYMED", "rela": r}
        res = get_rx_class_members(params)
        if res['related_ids']:
            rx_concepts.update(res['related_ids'])

print(f"Found {len(rx_concepts)} initial RxNorm concepts")

# Get related concepts
print("Finding related concepts...")
all_rx_list = list(rx_concepts)
count = 0
for rxcui in rx_concepts:
    res = get_related_rxcui(rxcui)
    if res['related_ids']:
        all_rx_list.extend(res['related_ids'])
    count += 1
    if count % 100 == 0:
        print(f"Processed {count}/{len(rx_concepts)} concepts...")

all_rx_final = list(set(all_rx_list))
print(f"Found {len(all_rx_final)} total RxNorm concepts after expansion")

# 5. Compare with existing VO annotations
print("Pulling RxNorm data for identified terms ...")
# rxnav_data = [get_rxnorm_data(rxcui) for rxcui in all_rx_final]
# rxnav_df = pd.DataFrame(rxnav_data)
# print(len(rxnav_df), "rows")
# change df column "names" to "label"
# rxnav_df.rename(columns={"name": "label"}, inplace=True)
# rxnav_df.to_pickle("rxnav_data.pkl")

Found 575 initial RxNorm concepts
Finding related concepts...
Processed 100/575 concepts...
Processed 200/575 concepts...
Processed 300/575 concepts...
Processed 400/575 concepts...
Processed 500/575 concepts...
Found 1979 total RxNorm concepts after expansion
Pulling RxNorm data for identified terms ...


In [6]:
all_rx_final

['1007132',
 '1593131',
 '2262062',
 '1928338',
 '1658470',
 '2270605',
 '1151126',
 '2647202',
 '2716813',
 '2718374',
 '2583742',
 '316950',
 '901508',
 '2636596',
 '2694006',
 '2605537',
 '2718399',
 '2273309',
 '1664171',
 '2658830',
 '2648283',
 '1601400',
 '1174360',
 '2562553',
 '2647726',
 '2714925',
 '836634',
 '2583638',
 '2664830',
 '803370',
 '2705457',
 '1007290',
 '2707129',
 '2663348',
 '2583640',
 '2583649',
 '2636598',
 '798295',
 '2673246',
 '2664725',
 '2479833',
 '2685009',
 '2705465',
 '2642309',
 '2646716',
 '2649382',
 '2718459',
 '2642299',
 '2646717',
 '2685003',
 '2468231',
 '1300379',
 '797635',
 '1942125',
 '832682',
 '2718382',
 '1190916',
 '2673241',
 '2719412',
 '1597095',
 '2707126',
 '2641827',
 '2648241',
 '798221',
 '1008344',
 '2583644',
 '2687362',
 '2659297',
 '2607745',
 '2656498',
 '854951',
 '797753',
 '1160330',
 '721656',
 '763097',
 '2647560',
 '2468233',
 '2656344',
 '798429',
 '1187406',
 '798229',
 '2593121',
 '2718469',
 '2718389',
 '1292

In [1]:
# change df column "names" to "label"
rxnav_df.rename(columns={"name": "label"}, inplace=True)
# count the number of unique RXCUIs
unique_rxcuis = rxnav_df['rxcui'].nunique()
print(f"Number of unique RXCUIs: {unique_rxcuis}")

NameError: name 'rxnav_df' is not defined

In [35]:
rxnav_df = pd.read_pickle("rxnav_data.pkl")
rxnav_df

,rxcui,label,status,tty
0,2566422,0.5 ML Streptococcus pneumoniae serotype 1 cap...,Active,SBD
1,2687724,Fluzone 2024-2025,Active,BN
2,1601400,Neisseria meningitidis serogroup B strain NZ98...,Active,SCDC
3,2270606,Vibrio cholerae CVD 103-HGR strain live antige...,Active,SCDG
4,1190919,"0.5 ML diphtheria toxoid vaccine, inactivated ...",Active,SBD
...,...,...,...,...
1879,2688113,influenza A virus A/Sydney/1304/2022 (H3N2) an...,Active,PIN
1880,2685010,Mresvia Injectable Product,Active,SBDG
1881,2270605,Vibrio cholerae CVD 103-HGR strain live antige...,Active,SCDG
1882,798231,Streptococcus pneumoniae serotype 6B capsular ...,Active,SCDC


#### 2.3 Retrival of RxNorm Obsolete Concepts

In [7]:
obsolete = get_all_obsolete_concepts()
obsolete_df = pd.DataFrame(obsolete)
obsolete_df['rxcui'] = obsolete_df['rxcui'].astype('Int64')
obsolete_df.rename(columns={"name": "label"}, inplace=True)

Found 97699 obsolete concepts.


In [30]:
model = load_model(MODEL_ID_LIST[0])

Loading model tavakolih/all-MiniLM-L6-v2-pubmed-full...
Model tavakolih/all-MiniLM-L6-v2-pubmed-full loaded to CUDA cuda:0 successfully.


In [33]:
# Obtaining Related Obsolete Concepts. Preliminary step using similarity. 
# ToDo: Load RxNorm DB (RXNREL) and identify related obsolete concepts
vaccine_texts = rxnav_df["label"].tolist()
obsolete_texts = obsolete_df["label"].tolist()
vaccine_vecs = encode_texts(model, vaccine_texts, batch_size=64, normalize=True)
obsolete_vecs = encode_texts(model, obsolete_texts, batch_size=64, normalize=True)
rx_faiss_ids = rxnav_df.index.to_numpy(dtype="int64")
rx_faiss_index = build_faiss_index(vaccine_vecs, ids=rx_faiss_ids, use_gpu=True)

k = 1
D, I = rx_faiss_index.search(obsolete_vecs, k)
similarity_threshold = 0.9
vaccine_like_obsolete = []

matches = []
for idx, (similarities, indices) in enumerate(zip(D, I)):
    for sim, i in zip(similarities, indices):
        if sim >= similarity_threshold:
            matches.append(obsolete_df.iloc[idx].get("rxcui"))

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Batches:   0%|          | 0/1527 [00:00<?, ?it/s]

Index built with 1884 IDs.


In [ ]:
rx_faiss_ids = build_and_save_dense_index(model,vaccine_texts,vaccine_vecs,index_name="rx_faiss_ids",save_path="rx_faiss_ids.pkl",use_gpu=True,)

In [34]:
len(matches), "matches found in obsolete_df"
# load obsolete_df where rxcui is in matches
ob_df = obsolete_df[obsolete_df['rxcui'].isin(matches)].copy()
# add a column "status" to ob_df
ob_df['status'] = 'obsolete'
ob_df.reset_index(drop=True, inplace=True)
ob_df

,rxcui,label,tty,status
0,1153962,"acellular pertussis vaccine, inactivated Injec...",SCDG,obsolete
1,1156379,"mumps virus vaccine live, Jeryl Lynn strain / ...",SCDG,obsolete
2,1156380,"mumps virus vaccine live, Jeryl Lynn strain In...",SCDG,obsolete
3,1156453,rubella virus vaccine live (Wistar RA 27-3 str...,SCDG,obsolete
4,1158981,"diphtheria toxoid vaccine, inactivated / Haemo...",SCDG,obsolete
...,...,...,...,...
1087,995112,hepatitis A vaccine (inactivated) strain HM175...,SCD,obsolete
1088,995160,hepatitis B surface antigen vaccine 0.0025 MG/ML,SCDC,obsolete
1089,995161,1 ML hepatitis B surface antigen vaccine 0.002...,SCD,obsolete
1090,995162,hepatitis B surface antigen vaccine 0.0025 MG/...,SBDC,obsolete


####  3. Loading and preprocessing VO 

In [ ]:
vo_df = pd.DataFrame
vo = get_ontology("vo_source/VO_merged2022_10_05.owl").load()
rows = []
for cls in vo.classes():
    name = cls.name
    if name.startswith("VO_"):
        label_text = cls.label.en.first() if isinstance(cls.label.first(), locstr) else cls.label.first()
        rxcui = cls.VO_0003198.first() if hasattr(cls, 'VO_0003198') and type(cls.VO_0003198.first())==int else None
        rows.append({"vo_id": name, "label": label_text, "rxcui": rxcui})
vo_df = pd.DataFrame(rows)
# drop rows with label that contains 'obsolete'
vo_df = vo_df[~vo_df['label'].str.contains("obsolete", case=False, na=False)]
# vo_df = vo_df.fillna("")
vo_df.to_pickle("vo_data.pkl")

In [36]:
# unpickle the dataframe
vo_df = pd.read_pickle("vo_data.pkl")

### Dense Index Setup

In [ ]:
texts = vo_df["label"].tolist()
vecs = encode_texts(model, texts, batch_size=64, normalize=True)
faiss_ids = vo_df.index.to_numpy(dtype="int64")  # Use DataFrame index as IDs
faiss_index = build_faiss_index(vecs, ids=faiss_ids, use_gpu=True)
faiss_cpu = faiss.index_gpu_to_cpu(faiss_index)
faiss.write_index(faiss_cpu, "vo_faiss.bin")

In [ ]:
build_and_save_dense_index(
    df=vo_df, 
    model=model,
    label_column="label",
    batch_size=64,
    normalize=True,
    use_gpu=True,
    save_index=True,
    index_filename="vo_faiss.bin"
)

In [ ]:
# del model, vo_vecs
# # clear cache
# torch.cuda.empty_cache()
# import gc
# gc.collect()

### BM25 Index Setup

In [37]:
def doc_actions(df, index_name="vo_bm25"):
    """
    df columns expected: ['vo_id', 'text'].
    ToDo: One row per VO label or synonym. 
    """
    df = df.fillna("")           

    for idx, row in df.iterrows():
        yield {
            "_op_type": "index",
            "_index"  : index_name,
            "_id"     : f"{row.vo_id}-{idx}",    # unique per synonym
            "vo_id"   : row.vo_id,
            # "iri"     : row.iri,
            "text"    : row.label
        }


In [38]:
from elasticsearch.helpers import streaming_bulk
for ok, action in streaming_bulk(
        es, doc_actions(vo_df), chunk_size=1000, max_retries=5, initial_backoff=2
):
    if not ok:
        print("Failed to index document:", action)

### Candidate Generation and Ranking

In [ ]:
def dense_candidates(faiss_index, q_vec, meta_df, k=20):
    D, I = faiss_index.search(q_vec.reshape(1,-1), k)
    rows = meta_df.loc[I[0], ["vo_id","label"]]
    rows = rows.assign(score=D[0])
    return rows.reset_index(drop=True).to_dict(orient="records")

In [ ]:
def bm25_candidates(es, rx_label: str, k: int = 20, index="vo_bm25"):
    """
    Return top-k lexical hits for a VO label from the RxNorm BM25 index.
    """
    resp = es.search(
        index=index,              # ← keyword
        size=k,                   # ← keyword
        query={                   # ← keyword
            "match": {
                "text": {         # field defined in mappings
                    "query": rx_label,
                    "operator": "or"
                }
            }
        }
    )
    return [
        {
            "vo_id": hit["_source"]["vo_id"],
            "label": hit["_source"]["text"],
            "score": hit["_score"]
        }
        for hit in resp["hits"]["hits"]
    ]

In [ ]:
# Reciprocal Rank Fusion
from itertools import chain
from collections import defaultdict

def fuse_hits_rrf(dense_hits, bm25_hits, k=20, rank_bias=60):

    # 1. sort each list by its native score (descending)
    dense_hits  = sorted(dense_hits,  key=lambda x: -x["score"])
    bm25_hits   = sorted(bm25_hits,   key=lambda x: -x["score"])

    # 2. compute RRF contribution from each list
    fused = defaultdict(lambda: {"rrf": 0.0, "origin": set()})

    for rank, hit in enumerate(dense_hits, start=1):
        key = hit["vo_id"]
        fused[key]["rrf"] += 1 / (rank_bias + rank)
        fused[key]["origin"].add("dense")
        # keep a representative label
        fused[key].setdefault("label", hit["label"])

    for rank, hit in enumerate(bm25_hits,  start=1):
        key = hit["vo_id"]
        fused[key]["rrf"] += 1 / (rank_bias + rank)
        fused[key]["origin"].add("bm25")
        fused[key].setdefault("label", hit["label"])

    # 3. turn dict -> list, sort by fused score
    fused_list = [
        {
            "vo_id": key,
            "label": val["label"],
            "fused": val["rrf"],
            "from" : ",".join(sorted(val["origin"]))   # e.g. "bm25,dense"
        }
        for key, val in fused.items()
    ]
    fused_list.sort(key=lambda x: -x["fused"])

    # 4. return top-k
    return fused_list[:k]


In [ ]:
# load the faiss index from file
faiss_cpu = faiss.read_index("vo_faiss.bin")
# load index to GPU
res = faiss.StandardGpuResources()
faiss_index = faiss.index_cpu_to_gpu(res, 0, faiss_cpu)

#### Smoke Testing

In [ ]:
# calculate cosine similarity for between two vectors
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

In [ ]:
query = rxnav_df.iloc[0].label
q_vec  = encode_texts(model, [query], batch_size=64, normalize=True)[0]    
t1 = vo_df[vo_df.index==3652].label.values[0]
s_vec = encode_texts(model, [t1], batch_size=64, normalize=True)[0]
cosine_sim = cosine_similarity(s_vec, q_vec)
print(f"Cosine similarity between '{query}' and the {t1} vector: {cosine_sim:.6f}")

In [ ]:
k = 10
n = 15
# select first row of rxnavdf
query = rxnav_df.iloc[0].label
q_vec  = encode_texts(model, [query], batch_size=64, normalize=True)
dense_hits = dense_candidates(faiss_index, q_vec, vo_df, k)
lexical_hits = bm25_candidates(es, query, k)
print(f"Lexical hits for '{query}':")
candidates = fuse_hits_rrf(dense_hits, lexical_hits, k=n)
candidates

### LLM refinement

In [ ]:
def generate_vo_candidates_for_rxnorm(
    rxnav_df: pd.DataFrame,
    vo_df: pd.DataFrame,
    model,
    faiss_index,
    es,
    k_dense=10,
    k_lexical=10,
    n_fused=15,
    batch_size=64,
    output_jsonl_path=None
):
    """
    Generate top VO candidates for each RxNorm concept using dense + lexical retrieval + RRF.

    Parameters:
        rxnav_df (pd.DataFrame): DataFrame of RxNorm concepts with at least a 'label' column.
        vo_df (pd.DataFrame): VO terms with at least 'id' and 'label'.
        model: Embedding model (e.g., SentenceTransformer).
        faiss_index: Prebuilt FAISS index of VO embeddings.
        es: Elasticsearch/BM25 engine for lexical search.
        k_dense (int): Number of dense candidates to retrieve.
        k_lexical (int): Number of lexical candidates to retrieve.
        n_fused (int): Number of final RRF candidates to keep.
        batch_size (int): Batch size for text encoding.
        output_jsonl_path (str): Optional path to store output as JSONL.

    Returns:
        List[Dict]: One dictionary per RxNorm row, including fused candidates.
    """


    results = []
    # Batchwise encoding
    queries = rxnav_df["label"].tolist()
    query_vectors = encode_texts(model, queries, batch_size=batch_size, normalize=True)
    rxcuis = rxnav_df.get("rxcui", [f"row_{i}" for i in range(len(rxnav_df))]).tolist()

    for (query,rxcui,q_vec) in tqdm(zip(queries, rxcuis, query_vectors), total=len(queries), desc="Generating candidates"):
        dense_hits = dense_candidates(faiss_index, q_vec.reshape(1,-1), vo_df, k=k_dense)
        lexical_hits = bm25_candidates(es, query, k=k_lexical)
        fused = fuse_hits_rrf(dense_hits, lexical_hits, k=n_fused)

        entry = {
            "rxcui": rxcui,
            "label": query,
            "candidates": fused
        }

        results.append(entry)

        # Optional streaming write
        if output_jsonl_path:
            with open(output_jsonl_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry) + "\n")

    return results


In [ ]:
res = generate_vo_candidates_for_rxnorm(
    rxnav_df=rxnav_df,
    vo_df=vo_df,
    model=model,
    faiss_index=faiss_index,
    es=es,
    k_dense=10,
    k_lexical=10,
    n_fused=15,
    output_jsonl_path="vo_candidates.jsonl"
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json

In [5]:
INFERENCE_MODEL_ID = ["Qwen/Qwen2-7B-Instruct", "google/medgemma-27b-text-it"]

In [4]:
def load_inference_model(model_id):
    print(f"Loading inference model {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True)
    print(f"Inference model {model_id} loaded successfully.")
    return model, tokenizer

In [ ]:
# MODEL_ID = "Qwen/Qwen2-7B-Instruct"
# inf_tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_ID, use_fast=True, trust_remote_code=True)
# inf_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, 
#     torch_dtype=torch.float16, 
#     device_map="auto", 
#     trust_remote_code=True
# )

In [ ]:
# # Future ToDo: Use ingredients, TTY, included in prompt
# def build_prompt(rxnorm, vo_candidates):
#     # --- System role ---
#     sys_msg = ("""\
# You are an ontology-alignment assistant. 
# Your task is to map a RxNorm clinical drug concept to the best matching Vaccine Ontology (VO) term.

# * Always choose the **single best** match if it exists.
# * If none of the candidates is an acceptable mapping, output "NONE".
# * Return answer as **valid JSON object** matching this schema:
#   {"VO_ID": "<vo_id | NONE>", "match_type": "exact|narrow|broad|none",
#    "justification": "<max 40 words>"}"""
#     )

#     # --- User message: Source concept ---
#     header = (f"""\
# ### RxNorm concept
# RxCUI: {rxnorm["rxcui"]}
# Label: {rxnorm["label"]}
# Ingredients: {", ".join(rxnorm.get("ingredients", [])) or "None"}
# Dose Form: {rxnorm.get("dose_form", "Unknown")}
# Route: {rxnorm.get("route", "Unknown")}
# Strength: {rxnorm.get("strength", "Unknown")}"""
#     )

#     # --- VO Candidates ---
#     cand_lines = [
#         f"{i}. {c['id']} | {c['label']} | score={c['fused']:.4f} | {c.get('definition','').strip()[:100]}"
#         for i, c in enumerate(vo_candidates, 1)
#     ]

#     # --- Instructions ---
#     user_msg = (
#         header +
#         "\n\n### VO Candidates\n" + "\n".join(cand_lines) +
#         "\n\n### Instructions\n"
#         "Choose the best VO concept **by VO ID**. Focus on antigen, formulation, dose, and usage context.\n"
#         "Respond with JSON only — no code fences, no extra keys."
#     )

#     return [
#         {"role": "system", "content": sys_msg},
#         {"role": "user", "content": user_msg}
#     ]


In [6]:
inference_model_id = INFERENCE_MODEL_ID[1]
inf_model, inf_tokenizer = load_inference_model(inference_model_id)

Loading inference model google/medgemma-27b-text-it...


Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

Inference model google/medgemma-27b-text-it loaded successfully.


In [13]:
def build_prompt(rxcui, rxlabel, vo_candidates):
    # --- System role ---
    sys_msg = ("""\
You are an ontology-alignment assistant. 
Your task is to map a RxNorm clinical drug concept to the best matching Vaccine Ontology (VO) term.

* Always choose the **single best** match if it exists.
* If none of the candidates is an acceptable mapping, output "NONE".
* Return answer as **valid JSON object** matching this schema EXACTLY:
  {"VO_ID": "<vo_id | NONE>", "match_type": "exact|narrow|broad|none",
   "justification": "<max 50 words>"}
* Do not include any other text, explanation, or code fences before or after the JSON."""
    )

    # --- User message: Source concept ---
    header = (f"""\
### RxNorm concept
RxCUI: {rxcui}
Label: {rxlabel}"""
    )

    # --- VO Candidates ---
    cand_lines = [
        f"{i}. {c['vo_id']} | {c['label']} | score={c['fused']:.4f}"
        for i, c in enumerate(vo_candidates, 1)
    ]

    # --- Instructions ---
    user_msg = (
        header +
        "\n\n### VO Candidates\n" + "\n".join(cand_lines) +
        "\n\n### Instructions\n"
        "Each candidate has a reciprocal rank fusion score derived from either embedding similarity (L2 distance) or lexical similarity (BM25). "
        "Higher scores indicate a stronger semantic match. Use this score as a guideline, but base your decision on "
        "vaccine-specific details like antigen, strain, formulation, dose, vaccine season, formulation, number of viral strains and route.\n\n"
        "Choose the best VO concept **by VO_ID**. Respond with JSON only — NO CODE FENCES, no extra keys."
)

    return [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]


In [ ]:
# chat = build_prompt(vo_term, candidates)

# prompt_ids = inf_tokenizer.apply_chat_template(
#     chat,
#     add_generation_prompt=True,   # inserts the assistant “turn header”
#     return_tensors="pt"
# ).to(inf_model.device)

# out = inf_model.generate(
#     prompt_ids,
#     max_new_tokens=120,
#     temperature=0.1,
#     repetition_penalty=1.1,
#     do_sample=False
# )

# assistant_text = inf_tokenizer.decode(out[0][prompt_ids.shape[-1]:],
#                                   skip_special_tokens=True).strip()
# mapping = json.loads(assistant_text)

In [ ]:
# import json

# def run_mapping_inference(vo_term, candidates, inf_tokenizer, inf_model):
#     """
#     Runs LLM-based mapping inference using the given term and candidate concepts.

#     Parameters:
#         vo_term (str): The source term from VO.
#         candidates (list[dict]): Candidate terms with attributes to map against.
#         inf_tokenizer: Tokenizer compatible with the inference model.
#         inf_model: A pre-loaded HuggingFace model supporting `.generate()`.

#     Returns:
#         dict: Parsed JSON mapping result from the model's assistant response.
#     """
#     chat = build_prompt(vo_term, candidates)

#     prompt_ids = inf_tokenizer.apply_chat_template(
#         chat,
#         add_generation_prompt=True,
#         return_tensors="pt"
#     ).to(inf_model.device)

#     out = inf_model.generate(
#         prompt_ids,
#         max_new_tokens=120,
#         temperature=0.1,
#         repetition_penalty=1.1,
#         do_sample=False
#     )

#     assistant_text = inf_tokenizer.decode(
#         out[0][prompt_ids.shape[-1]:],
#         skip_special_tokens=True
#     ).strip()

#     return json.loads(assistant_text)


In [1]:
def run_mapping_inference(rxcui, rxlabel, vo_candidates, inf_tokenizer, inf_model):
    """
    Runs LLM-based mapping inference to align an RxNorm concept with the best-matching VO term.

    Parameters:
        rxcui (str): RxCUI of the source RxNorm concept.
        rxlabel (str): Label of the source RxNorm concept.
        vo_candidates (list[dict]): Candidate VO terms with IDs, labels, definitions, and similarity scores.
        inf_tokenizer: HuggingFace tokenizer with `apply_chat_template`.
        inf_model: HuggingFace model that supports `.generate()`.

    Returns:
        dict: Parsed JSON mapping result with keys:
              - rxcui (or "NONE")
              - match_type (exact | narrow | broad | none)
              - justification (short rationale)
    """
    chat = build_prompt(rxcui, rxlabel, vo_candidates)
    text = inf_tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        tokenize=False
    )
    model_inputs = inf_tokenizer([text], return_tensors="pt", padding=True).to(inf_model.device)
    if inf_tokenizer.pad_token is None:
        inf_tokenizer.pad_token = inf_tokenizer.eos_token

    generated_ids = inf_model.generate(
        model_inputs.input_ids,
        pad_token_id=inf_tokenizer.pad_token_id,
        max_new_tokens=500,
        # repetition_penalty=1.1,
        # do_sample=False
    )
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    assistant_text = inf_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Try to extract JSON from the response
    try:
        # First attempt: parse as-is
        return json.loads(assistant_text)
    except json.JSONDecodeError:
        # Second attempt: find JSON-like patterns with regex
        json_pattern = r'(\{.*\})'
        match = re.search(json_pattern, assistant_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                pass
        
        # Fallback: return structured error response
        print(f"Failed to parse JSON from: {assistant_text}...")
        return {
            "VO_ID": "NONE",
            "match_type": "none",
            "justification": "Error parsing model output",
            "error": "JSONDecodeError"
        }

    # return json.loads(assistant_text)

In [15]:
# Iterate over rxnav_df
candidates = '/data/wmanuel3/VaxMapperRepo/vo_candidates.jsonl'
# mapping = []
output_jsonl_path=f"inference_results{inference_model_id.replace('/', '_')}_1.jsonl"
i= 0
with open(candidates, 'r', encoding='utf-8') as f:
    for line in f:
        i += 1
        entry = json.loads(line)
        rxcui = entry['rxcui']
        rxlabel = entry['label']
        vo_candidates = entry['candidates']
        
        # Run inference
        result = run_mapping_inference(
            rxcui=rxcui,
            rxlabel=rxlabel,
            vo_candidates=vo_candidates,
            inf_tokenizer=inf_tokenizer,
            inf_model=inf_model
        )
        result['rxcui'] = rxcui
        result['label'] = rxlabel
        # mapping.append(result)
        with open(output_jsonl_path, 'a', encoding='utf-8') as out_f:
            out_f.write(json.dumps(result) + "\n")
        # if i == 100:
        #     break
    

KeyboardInterrupt: 

In [ ]:
result

In [ ]:
vo_df[vo_df['vo_id'] == 'VO_0003824']

In [ ]:
output_jsonl_path="inference_results.jsonl"

In [ ]:
# read jsonl file into a dataframe using only VO_ID and rxcui
import pandas as pd
inference_df = pd.read_json(output_jsonl_path, lines=True)
inference_df = inference_df[['rxcui', 'VO_ID']]
inference_df.rename(columns={'VO_ID': 'vo_id'}, inplace=True)
inference_df['rxcui'] = inference_df['rxcui'].astype(int)
inference_df


In [6]:
vo_df = pd.read_pickle("vo_data.pkl")
vo_df

,vo_id,label,rxcui
0,VO_0000001,vaccine,NaN
2,VO_0000599,recombinant vaccine vector,NaN
3,VO_0000574,route of administration,NaN
4,VO_0000333,Protozoa,NaN
5,VO_0001361,virmugen role,NaN
...,...,...,...
6226,VO_0015053,"Measles Virus Vaccine Live, Enders' attenuated...",804180.0
6227,VO_0015058,"Mumps Virus Vaccine Live, Jeryl Lynn Strain In...",1156380.0
6228,VO_0015059,"Mumps Virus Vaccine Live, Jeryl Lynn Strain 40...",798375.0
6229,VO_0015060,"Mumps Virus Vaccine Live, Jeryl Lynn Strain 25...",804181.0


In [ ]:
rxnav_df= pd.read_pickle("rxnav_data.pkl")
rxnav_df = rxnav_df[['rxcui', 'label']]
rxnav_df['rxcui'] = rxnav_df['rxcui'].astype('Int64')
rxnav_df

In [ ]:
vo_expanded_df = pd.merge(vo_df, rxnav_df, on='rxcui', how='left', suffixes=('_vo', '_rxnav'))
vo_expanded_df['exact'] = vo_expanded_df.apply(lambda x: x['label_vo'] == x['label_rxnav'], axis=1)
vo_expanded_df

In [ ]:
#check if inference_df has predicted the correct rxcui (from vo_df)
vo_df['rxcui'] = vo_df['rxcui'].astype('Int64')
merged_df = pd.merge(vo_df, inference_df, on='vo_id', how='left', suffixes=('_vo', '_inference'))
# # check if vo_id from vo_df is in VO_ID from inference_df
merged_df['correct'] = merged_df.apply(lambda x: x['rxcui_vo'] == x['rxcui_inference'], axis=1)
# # count how many correct
correct_count = merged_df['correct'].sum()
total_count = len(vo_df[vo_df['rxcui'].notnull()])
accuracy = correct_count / total_count * 100

In [ ]:
merged_df

In [ ]:
merged_df[merged_df['correct'] == True]

In [ ]:
# def evaluate_mapping_accuracy(
#     predictions: dict,
#     gold_df: pd.DataFrame,
#     top_n: int = 5
# ):
#     """
#     Evaluate Top-N accuracy of RxNorm → VO candidate predictions.

#     Parameters:
#         predictions (dict): Mapping of RxCUI → list of candidate VO dicts (with 'id' key), e.g.
#             {
#               "12345": [{"id": "VO_0001", "score": 0.93}, ...],
#               ...
#             }
#         gold_df (pd.DataFrame): DataFrame with columns ['rxcui', 'vo_id'] (ground truth).
#         top_n (int): Number of top candidates to consider for match.

#     Returns:
#         accuracy (float): Proportion of RxCUIs with a correct match in top-N.
#         matched_rxcuis (List[str]): RxCUIs that were correctly matched.
#         missed_rxcuis (List[str]): RxCUIs that were not matched.
#     """
#     gold_map = dict(zip(gold_df.rxcui.astype(str), gold_df.vo_id))

#     correct = 0
#     matched = []
#     missed = []

#     for rxcui, candidates in predictions.items():
#         gold_vo = gold_map.get(str(rxcui))
#         if not gold_vo:
#             continue  # Skip if gold mapping not available

#         top_candidate_ids = [c["id"] for c in candidates[:top_n]]
#         if gold_vo in top_candidate_ids:
#             correct += 1
#             matched.append(rxcui)
#         else:
#             missed.append(rxcui)

#     total = len(gold_map)
#     accuracy = correct / total if total > 0 else 0.0
#     return accuracy, matched, missed


In [7]:
import json
import pandas as pd

def evaluate_topn_accuracy_from_jsonl(predictions_path, gold_df, top_n=5):
    """
    Evaluate Top-N accuracy of RxNorm → VO candidate predictions from a JSONL file.

    Parameters:
        predictions_path (str): Path to JSONL file with predictions.
        gold_df (pd.DataFrame): Gold standard with columns ['rxcui', 'vo_id'].
        top_n (int): Number of top candidates to consider for evaluation.

    Returns:
        accuracy (float): Top-N accuracy.
        matched (list[str]): RxCUIs correctly matched.
        missed (list[str]): RxCUIs not matched in top-N.
    """
    gold_map = dict(zip(gold_df.rxcui.astype(str), gold_df.vo_id))
    
    matched = []
    missed = []
    total = 0

    with open(predictions_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            rxcui = str(entry['rxcui'])
            predicted_vo_ids = [cand['vo_id'] for cand in entry['candidates'][:top_n]]

            if rxcui in gold_map:
                total += 1
                if gold_map[rxcui] in predicted_vo_ids:
                    matched.append(rxcui)
                else:
                    missed.append(rxcui)

    accuracy = len(matched) / total if total else 0.0
    return accuracy, matched, missed


In [ ]:
gold_df = pd.read_pickle("vo_data.pkl")
accuracy, matched, missed = evaluate_topn_accuracy_from_jsonl('vo_candidates.jsonl', gold_df, top_n=5)
print(f"Top-5 Accuracy: {accuracy:.2%}")
print(f"Matched: {len(matched)} | Missed: {len(missed)}")

In [ ]:
print(f"Accuracy of VO inference: {accuracy:.2f}% ({correct_count}/{total_count})") 

### 4. Combined Pipeline and Evaluation: 

In [ ]:
#Evaluation of hybrid mappings (embedding + BM25 candidates)
# open vo_candidates.jsonl and read it into a dataframe
vo_candidates_df = pd.read_json(candidates, lines=True)
vo_candidates_df = vo_candidates_df.explode('candidates')
vo_candidates_df['vo_id'] = vo_candidates_df['candidates'].apply(lambda x: x['vo_id'])
vo_candidates_df

,rxcui,label,candidates,vo_id
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,"{'vo_id': 'VO_0003678', 'label': 'Fluvirin 201...",VO_0003678
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,"{'vo_id': 'VO_0003683', 'label': 'influenza A ...",VO_0003683
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,"{'vo_id': 'VO_0003684', 'label': 'Fluvirin 201...",VO_0003684
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,"{'vo_id': 'VO_0003700', 'label': 'influenza A ...",VO_0003700
0,2647547,influenza A virus A/Christchurch/16/2010 (H1N1...,"{'vo_id': 'VO_0003583', 'label': 'Fluzone 2015...",VO_0003583
...,...,...,...,...
5651,2647703,respiratory syncytial virus pre-fusion F prote...,"{'vo_id': 'VO_0003370', 'label': 'Neisseria me...",VO_0003370
5651,2647703,respiratory syncytial virus pre-fusion F prote...,"{'vo_id': 'VO_0001429', 'label': 'Bovine Respi...",VO_0001429
5651,2647703,respiratory syncytial virus pre-fusion F prote...,"{'vo_id': 'VO_0003361', 'label': 'Neisseria me...",VO_0003361
5651,2647703,respiratory syncytial virus pre-fusion F prote...,"{'vo_id': 'VO_0002330', 'label': 'Bovine Parai...",VO_0002330


In [ ]:
vo_df['rxcui'] = vo_df['rxcui'].astype('Int64')
merged_can_df = pd.merge(vo_df, vo_candidates_df, on='vo_id', how='left', suffixes=('_vo', '_candidates'))
merged_can_df['correct'] = merged_can_df.apply(lambda x: x['rxcui_vo'] == x['rxcui_candidates'], axis=1)
correct_by_vo = merged_can_df.groupby('vo_id')['correct'].any().reset_index()
correct_count = correct_by_vo['correct'].sum()
total_count = len(vo_df[vo_df['rxcui'].notnull()])
accuracy = correct_count / total_count * 100
print(f"Accuracy of VO inference: {accuracy:.2f}% ({correct_count}/{total_count})") 

Accuracy of VO inference: 50.84% (455/895)


In [ ]:
## About the direction of the new concept placement: depends on the context. 

In [23]:
correct_by_vo

,vo_id,correct
0,VO_0000000,False
1,VO_0000001,False
2,VO_0000002,False
3,VO_0000003,True
4,VO_0000004,False
...,...,...
6217,VO_0015081,False
6218,VO_0015084,True
6219,VO_0015085,True
6220,VO_0015086,True


### 5. Agentic Workflow

In [1]:
import networkx as nx
vo = get_ontology("vo_source/VO_merged2022_10_05.owl").load()
def get_ontology_graph(vo):
    G = nx.DiGraph()
    for cls in vo.classes():
        if cls.is_a:  # has parents
            for parent in cls.is_a:
                if isinstance(parent, owlready2.ThingClass):
                    G.add_edge(parent.name, cls.name)
    return G
vo_graph = get_ontology_graph(vo)

NameError: name 'get_ontology' is not defined

In [ ]:
from langchain.agents import initialize_agent
from langchain.tools import Tool

# tools = [
#     Tool(
#         name="search",
#         func=search,
#         description="Search for information"
#     ),
#     Tool(
#         name="lookup",
#         func=lookup,
#         description="Look up information"
#     )
# ]

# agent = initialize_agent(tools)

<!-- Evaluation of existing mapping -
Evaluation of new mapping (between version) 
Evaluation of missing concept mapping -  -->

In [ ]:
# Testing of the general (Cell line ontology)
# missing is-a relationships
